In [1]:
#If working locally, download the .py file and execute the following command in cmd to install required libraries
#pip install torch torchvision pillow opencv-python numpy

In [2]:
import torch
from torchvision import models, transforms
from PIL import Image
import cv2
import numpy as np
import urllib.request
import time
import os

In [3]:


if not os.path.exists("imagenet_classes.txt"):
    print("Downloading class labels...")
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/pytorch/hub/master/imagenet_classes.txt",
        "imagenet_classes.txt"
    )

with open("imagenet_classes.txt", "r") as f:
    categories = [s.strip() for s in f.readlines()]



In [4]:
#MODEL SETUP
print("Loading pre-trained ResNet18 model...")
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
model.eval()

transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

def predict(image_path):

    img = Image.open(image_path).convert("RGB")
    input_tensor = transform(img).unsqueeze(0)

    # TODO 4: Add timing logic here to measure inference latency (in milliseconds)
    # Start timer
    start_time = time.perf_counter()

    with torch.no_grad():
        output = model(input_tensor)

    # End timer
    end_time = time.perf_counter()

    # Calculate latency in milliseconds (seconds * 1000)
    latency_ms = (end_time - start_time) * 1000.0

    probs = torch.nn.functional.softmax(output[0], dim=0)
    confidence, predicted_idx = torch.max(probs, 0)



    return categories[predicted_idx.item()], confidence.item(), latency_ms



Loading pre-trained ResNet18 model...


Implement the following functions:

1.   List item
2.   List item



In [5]:
def simulate_turbidity(img_array):
    """
    Simulate murky water (blur/haze).
    Input: OpenCV image array (BGR)
    Output: Modified OpenCV image array
    """
    # TODO 1: Implement blur or haze using cv2 or numpy
    kernel_size = (31, 31)

    # Apply Gaussian blur
    img_array= cv2.GaussianBlur(img_array, kernel_size, 0)
    return img_array


In [6]:
import cv2
import numpy as np

def simulate_color_shift(img_array):
    """
    Simulate depth color loss (attenuate the red channel).
    Input: OpenCV image array (BGR)
    Output: Modified OpenCV image array
    """


    # 2. Extract the Red channel (Index 2 in BGR) and convert to float32
    red_channel = img_array[:, :, 2].astype(np.float32)

    # 3. Apply a fixed attenuation (e.g., 0.3 leaves 30% of the red light)
    red_channel = red_channel * 0.2

    # 4. Clip values to stay within 0-255 and cast back to uint8
    img_array[:, :, 2] = np.clip(red_channel, 0, 255).astype(np.uint8)

    return img_array

In [7]:
def simulate_sensor_noise(img_array):
    """
    Simulate low-light digital camera noise.
    Input: OpenCV image array (BGR)
    Output: Modified OpenCV image array
    """
    # 1. Set noise parameters internally
    mean = 0
    std_dev = 50  # Higher numbers = heavier grain/noise

    # 2. Generate random Gaussian noise matching the image's exact dimensions
    noise = np.random.normal(mean, std_dev, img_array.shape)

    # 3. Convert image to float32 to safely add negative and positive noise
    # This prevents math errors (like 250 + 10 wrapping around to 4 in uint8)
    img_array= img_array.astype(np.float32) + noise

    # 4. Clip the results to ensure no pixel falls below 0 or goes above 255
    img_array = np.clip(img_array, 0, 255)

    # 5. Cast the final array back to standard 8-bit format
    return img_array.astype(np.uint8)

In [8]:
if __name__ == "__main__":

    #change the base image address as required to test for all images
    base_image = "my_test_images/aquarium.jpg"
    img = cv2.imread(base_image)


    cv2.imwrite("test_turbid_aquarium.jpg", simulate_turbidity(img.copy()))
    cv2.imwrite("test_colorshift_aquarium.jpg", simulate_color_shift(img.copy()))
    cv2.imwrite("test_noise_aquarium.jpg", simulate_sensor_noise(img.copy()))

    images_to_test = [
        ("Baseline (Clean)", base_image),
        ("Turbidity", "test_turbid_aquarium.jpg"),
        ("Color Shift", "test_colorshift_aquarium.jpg"),
        ("Sensor Noise", "test_noise_aquarium.jpg")
    ]


    print("\nRUNNING VISION DIAGNOSTICS")


    for condition_name, file_path in images_to_test:
        label, conf, latency = predict(file_path)
        print(f"Condition : {condition_name}")
        print(f"Prediction: {label}")
        print(f"Confidence: {conf:.4f}")
        print(f"Latency   : {latency:.2f} ms")
        print("-" * 30)



RUNNING VISION DIAGNOSTICS
Condition : Baseline (Clean)
Prediction: eel
Confidence: 0.4804
Latency   : 58.79 ms
------------------------------
Condition : Turbidity
Prediction: brambling
Confidence: 0.0456
Latency   : 33.21 ms
------------------------------
Condition : Color Shift
Prediction: eel
Confidence: 0.6180
Latency   : 32.85 ms
------------------------------
Condition : Sensor Noise
Prediction: coral reef
Confidence: 0.0855
Latency   : 34.61 ms
------------------------------


### Result for aquarium

|            Baseline (Clean)             |            Turbidity (Murky)             |            Color Shift (Depth Loss)            |          Sensor Noise (Low Light)          |
|:---------------------------------------:|:----------------------------------------:|:----------------------------------------------:|:------------------------------------------:|
| ![Clean](./my_test_images/aquarium.jpg) | ![Turbidity](./test_turbid_aquarium.jpg) | ![Color Shift](./test_colorshift_aquarium.jpg) | ![Sensor Noise](./test_noise_aquarium.jpg) |

In [9]:
if __name__ == "__main__":

    #change the base image address as required to test for all images
    base_image = "my_test_images/jellyfish.jpg"
    img = cv2.imread(base_image)


    cv2.imwrite("test_turbid_jellyfish.jpg", simulate_turbidity(img.copy()))
    cv2.imwrite("test_colorshift_jellyfish.jpg", simulate_color_shift(img.copy()))
    cv2.imwrite("test_noise_jellyfish.jpg", simulate_sensor_noise(img.copy()))

    images_to_test = [
        ("Baseline (Clean)", base_image),
        ("Turbidity", "test_turbid_jellyfish.jpg"),
        ("Color Shift", "test_colorshift_jellyfish.jpg"),
        ("Sensor Noise", "test_noise_jellyfish.jpg")
    ]


    print("\nRUNNING VISION DIAGNOSTICS")


    for condition_name, file_path in images_to_test:
        label, conf, latency = predict(file_path)
        print(f"Condition : {condition_name}")
        print(f"Prediction: {label}")
        print(f"Confidence: {conf:.4f}")
        print(f"Latency   : {latency:.2f} ms")
        print("-" * 30)



RUNNING VISION DIAGNOSTICS
Condition : Baseline (Clean)
Prediction: jellyfish
Confidence: 0.9949
Latency   : 25.52 ms
------------------------------
Condition : Turbidity
Prediction: goldfish
Confidence: 0.1528
Latency   : 25.52 ms
------------------------------
Condition : Color Shift
Prediction: jellyfish
Confidence: 0.9940
Latency   : 21.18 ms
------------------------------
Condition : Sensor Noise
Prediction: jellyfish
Confidence: 0.1386
Latency   : 24.93 ms
------------------------------


### Result for jellyfish

| Baseline (Clean) |             Turbidity (Murky)             |            Color Shift (Depth Loss)             |          Sensor Noise (Low Light)           |
| :---: |:-----------------------------------------:|:-----------------------------------------------:|:-------------------------------------------:|
| ![Clean](./my_test_images/jellyfish.jpg) | ![Turbidity](./test_turbid_jellyfish.jpg) | ![Color Shift](./test_colorshift_jellyfish.jpg) | ![Sensor Noise](./test_noise_jellyfish.jpg) |

In [10]:
if __name__ == "__main__":

    #change the base image address as required to test for all images
    base_image = "my_test_images/set_f20_SESR.png"
    img = cv2.imread(base_image)


    cv2.imwrite("test_turbid_set_f20_SESR.jpg", simulate_turbidity(img.copy()))
    cv2.imwrite("test_colorshift_set_f20_SESR.jpg", simulate_color_shift(img.copy()))
    cv2.imwrite("test_noise_set_f20_SESR.jpg", simulate_sensor_noise(img.copy()))

    images_to_test = [
        ("Baseline (Clean)", base_image),
        ("Turbidity", "test_turbid_set_f20_SESR.jpg"),
        ("Color Shift", "test_colorshift_set_f20_SESR.jpg"),
        ("Sensor Noise", "test_noise_set_f20_SESR.jpg")
    ]


    print("\nRUNNING VISION DIAGNOSTICS")


    for condition_name, file_path in images_to_test:
        label, conf, latency = predict(file_path)
        print(f"Condition : {condition_name}")
        print(f"Prediction: {label}")
        print(f"Confidence: {conf:.4f}")
        print(f"Latency   : {latency:.2f} ms")
        print("-" * 30)



RUNNING VISION DIAGNOSTICS
Condition : Baseline (Clean)
Prediction: hen-of-the-woods
Confidence: 0.2156
Latency   : 24.90 ms
------------------------------
Condition : Turbidity
Prediction: coral fungus
Confidence: 0.1022
Latency   : 20.29 ms
------------------------------
Condition : Color Shift
Prediction: coral reef
Confidence: 0.3028
Latency   : 25.36 ms
------------------------------
Condition : Sensor Noise
Prediction: coral reef
Confidence: 0.1446
Latency   : 22.41 ms
------------------------------


### Result for set_f20_SESR.png

|              Baseline (Clean)               |              Turbidity (Murky)               |              Color Shift (Depth Loss)              |            Sensor Noise (Low Light)            |
|:-------------------------------------------:|:--------------------------------------------:|:--------------------------------------------------:|:----------------------------------------------:|
| ![Clean](./my_test_images/set_f20_SESR.png) | ![Turbidity](./test_turbid_set_f20_SESR.jpg) | ![Color Shift](./test_colorshift_set_f20_SESR.jpg) | ![Sensor Noise](./test_noise_set_f20_SESR.jpg) |

In [11]:
if __name__ == "__main__":

    #change the base image address as required to test for all images
    base_image = "my_test_images/set_f46_SESR.png"
    img = cv2.imread(base_image)


    cv2.imwrite("test_turbid_set_f46_SESR.jpg", simulate_turbidity(img.copy()))
    cv2.imwrite("test_colorshift_set_f46_SESR.jpg", simulate_color_shift(img.copy()))
    cv2.imwrite("test_noise_set_f46_SESR.jpg", simulate_sensor_noise(img.copy()))

    images_to_test = [
        ("Baseline (Clean)", base_image),
        ("Turbidity", "test_turbid_set_f46_SESR.jpg"),
        ("Color Shift", "test_colorshift_set_f46_SESR.jpg"),
        ("Sensor Noise", "test_noise_set_f46_SESR.jpg")
    ]


    print("\nRUNNING VISION DIAGNOSTICS")


    for condition_name, file_path in images_to_test:
        label, conf, latency = predict(file_path)
        print(f"Condition : {condition_name}")
        print(f"Prediction: {label}")
        print(f"Confidence: {conf:.4f}")
        print(f"Latency   : {latency:.2f} ms")
        print("-" * 30)



RUNNING VISION DIAGNOSTICS
Condition : Baseline (Clean)
Prediction: king crab
Confidence: 0.3312
Latency   : 18.38 ms
------------------------------
Condition : Turbidity
Prediction: tarantula
Confidence: 0.4672
Latency   : 24.47 ms
------------------------------
Condition : Color Shift
Prediction: king crab
Confidence: 0.6450
Latency   : 22.21 ms
------------------------------
Condition : Sensor Noise
Prediction: tarantula
Confidence: 0.5890
Latency   : 19.50 ms
------------------------------


### Result for .png

|              Baseline (Clean)               |              Turbidity (Murky)               |              Color Shift (Depth Loss)              |            Sensor Noise (Low Light)            |
|:-------------------------------------------:|:--------------------------------------------:|:--------------------------------------------------:|:----------------------------------------------:|
| ![Clean](./my_test_images/set_f46_SESR.png) | ![Turbidity](./test_turbid_set_f46_SESR.jpg) | ![Color Shift](./test_colorshift_set_f46_SESR.jpg) | ![Sensor Noise](./test_noise_set_f46_SESR.jpg) |

In [12]:
if __name__ == "__main__":

    #change the base image address as required to test for all images
    base_image = "my_test_images/set_o20_SESR.png"
    img = cv2.imread(base_image)


    cv2.imwrite("test_turbid_set_o20_SESR.jpg", simulate_turbidity(img.copy()))
    cv2.imwrite("test_colorshift_set_o20_SESR.jpg", simulate_color_shift(img.copy()))
    cv2.imwrite("test_noise_set_o20_SESR.jpg", simulate_sensor_noise(img.copy()))

    images_to_test = [
        ("Baseline (Clean)", base_image),
        ("Turbidity", "test_turbid_set_o20_SESR.jpg"),
        ("Color Shift", "test_colorshift_set_o20_SESR.jpg"),
        ("Sensor Noise", "test_noise_set_o20_SESR.jpg")
    ]


    print("\nRUNNING VISION DIAGNOSTICS")


    for condition_name, file_path in images_to_test:
        label, conf, latency = predict(file_path)
        print(f"Condition : {condition_name}")
        print(f"Prediction: {label}")
        print(f"Confidence: {conf:.4f}")
        print(f"Latency   : {latency:.2f} ms")
        print("-" * 30)



RUNNING VISION DIAGNOSTICS
Condition : Baseline (Clean)
Prediction: loggerhead
Confidence: 0.7938
Latency   : 23.65 ms
------------------------------
Condition : Turbidity
Prediction: loggerhead
Confidence: 0.5158
Latency   : 24.28 ms
------------------------------
Condition : Color Shift
Prediction: loggerhead
Confidence: 0.6235
Latency   : 23.46 ms
------------------------------
Condition : Sensor Noise
Prediction: leatherback turtle
Confidence: 0.4252
Latency   : 23.75 ms
------------------------------


### Result for set_o20_SESR.png

|              Baseline (Clean)               |        Turbidity (Murky)         |        Color Shift (Depth Loss)        |      Sensor Noise (Low Light)      |
|:-------------------------------------------:|:--------------------------------:|:--------------------------------------:|:----------------------------------:|
| ![Clean](./my_test_images/set_o20_SESR.png) | ![Turbidity](./test_turbid_set_o20_SESR.jpg) | ![Color Shift](./test_colorshift_set_o20_SESR.jpg) | ![Sensor Noise](./test_noise_set_o20_SESR.jpg) |

In [13]:
if __name__ == "__main__":

    #change the base image address as required to test for all images
    base_image = "my_test_images/set_u106_SESR.png"
    img = cv2.imread(base_image)


    cv2.imwrite("test_turbid_set_u106_SESR.jpg", simulate_turbidity(img.copy()))
    cv2.imwrite("test_colorshift_set_u106_SESR.jpg", simulate_color_shift(img.copy()))
    cv2.imwrite("test_noise_set_u106_SESR.jpg", simulate_sensor_noise(img.copy()))

    images_to_test = [
        ("Baseline (Clean)", base_image),
        ("Turbidity", "test_turbid_set_u106_SESR.jpg"),
        ("Color Shift", "test_colorshift_set_u106_SESR.jpg"),
        ("Sensor Noise", "test_noise_set_u106_SESR.jpg")
    ]


    print("\nRUNNING VISION DIAGNOSTICS")


    for condition_name, file_path in images_to_test:
        label, conf, latency = predict(file_path)
        print(f"Condition : {condition_name}")
        print(f"Prediction: {label}")
        print(f"Confidence: {conf:.4f}")
        print(f"Latency   : {latency:.2f} ms")
        print("-" * 30)



RUNNING VISION DIAGNOSTICS
Condition : Baseline (Clean)
Prediction: rock beauty
Confidence: 0.8001
Latency   : 30.98 ms
------------------------------
Condition : Turbidity
Prediction: rock beauty
Confidence: 0.2477
Latency   : 20.81 ms
------------------------------
Condition : Color Shift
Prediction: rock beauty
Confidence: 0.8690
Latency   : 21.10 ms
------------------------------
Condition : Sensor Noise
Prediction: rock beauty
Confidence: 0.9152
Latency   : 31.60 ms
------------------------------


### Result for set_u106_SESR.png

|              Baseline (Clean)               |        Turbidity (Murky)         |        Color Shift (Depth Loss)        |      Sensor Noise (Low Light)      |
|:-------------------------------------------:|:--------------------------------:|:--------------------------------------:|:----------------------------------:|
| ![Clean](./my_test_images/set_u106_SESR.png) | ![Turbidity](./test_turbid_set_u106_SESR.jpg) | ![Color Shift](./test_colorshift_set_u106_SESR.jpg) | ![Sensor Noise](./test_noise_set_u106_SESR.jpg) |

In [14]:
if __name__ == "__main__":

    #change the base image address as required to test for all images
    base_image = "my_test_images/set_u113_SESR.png"
    img = cv2.imread(base_image)


    cv2.imwrite("test_turbid_set_u113_SESR.jpg", simulate_turbidity(img.copy()))
    cv2.imwrite("test_colorshift_set_u113_SESR.jpg", simulate_color_shift(img.copy()))
    cv2.imwrite("test_noise_set_u113_SESR.jpg", simulate_sensor_noise(img.copy()))

    images_to_test = [
        ("Baseline (Clean)", base_image),
        ("Turbidity", "test_turbid_set_u113_SESR.jpg"),
        ("Color Shift", "test_colorshift_set_u113_SESR.jpg"),
        ("Sensor Noise", "test_noise_set_u113_SESR.jpg")
    ]


    print("\nRUNNING VISION DIAGNOSTICS")


    for condition_name, file_path in images_to_test:
        label, conf, latency = predict(file_path)
        print(f"Condition : {condition_name}")
        print(f"Prediction: {label}")
        print(f"Confidence: {conf:.4f}")
        print(f"Latency   : {latency:.2f} ms")
        print("-" * 30)



RUNNING VISION DIAGNOSTICS
Condition : Baseline (Clean)
Prediction: king crab
Confidence: 0.2095
Latency   : 22.06 ms
------------------------------
Condition : Turbidity
Prediction: barn spider
Confidence: 0.2184
Latency   : 19.25 ms
------------------------------
Condition : Color Shift
Prediction: leatherback turtle
Confidence: 0.1820
Latency   : 19.32 ms
------------------------------
Condition : Sensor Noise
Prediction: leatherback turtle
Confidence: 0.3602
Latency   : 24.33 ms
------------------------------


### Result for set_u113_SESR.png

|              Baseline (Clean)               |        Turbidity (Murky)         |        Color Shift (Depth Loss)        |      Sensor Noise (Low Light)      |
|:-------------------------------------------:|:--------------------------------:|:--------------------------------------:|:----------------------------------:|
| ![Clean](./my_test_images/set_u113_SESR.png) | ![Turbidity](./test_turbid_set_u113_SESR.jpg) | ![Color Shift](./test_colorshift_set_u113_SESR.jpg) | ![Sensor Noise](./test_noise_set_u113_SESR.jpg) |